# YosysTestCase configuration

The behaviour of the test is configured using the class member `_yosys_params_`, an instance of the class `YosysParams`. During a test run, the library creates a SymbiYosys project based on these settings. Check the documentation of the `.sby` format for more details: https://symbiyosys.readthedocs.io/en/latest/reference.html

## parameters

This section describes the most important parameters of `YosysParams`. A complete list can be found in the doc-string of the class.

### build_dir / clean_build_dir / use_tmp_dir

These parameters define where the Yosys project is created and whether it is cleared before each test run.

### bmc / bmc_depth

When `bmc` is set to true, the solver performs a bounded model check. It tries to find any violated assertions. The test passes if no assertion fails for the first `bmc_depth` clock cycles.

### cover / cover_depth

When `cover` is set to true, the solver tries to find a solution for every cover statement. The test passes if every cover statement is reached within the first `cover_depth` clock cycles.

### prove / prove_depth

When running in prove mode the solver attempts to prove that the design works forever.

The zipcpu blog has some great posts on the topic: https://zipcpu.com/blog/2017/10/19/formal-intro.html

### live

When `live` is set to true, a liveness check is performed. This feature is untested in cohdl_yosys because it is only supported by the `aiger suprove` engine and I did not manage to get that to work.

### engines

As described in the `.sby` reference linked above, SymbiYosys supports different solvers/engines. The strings passed to the `engines` parameter are added to the `[engines]` section of the configuration file.

## inheritance

`_yosys_params_` is not used directly. Instead an effective set of parameters is generated based on the settings in all parent classes. For each parameter the value specified in the most derived class is used.

The intension behind this is to make it possible to define common settings in a base class.

In the example below the parameters of `FormalBase` is inherited by `Formal_A` and `Formal_B`. `Formal_B` overwrites the setting of `bmc_depth` with a custom value.

In [1]:
import cohdl
from cohdl_yosys import YosysTestCase, YosysParams

class ExampleEntity(cohdl.Entity):
    ...

class FormalBase(YosysTestCase):
    """ a reusable base entity that defines common parameters """

    _yosys_params_ = YosysParams(
        # create a temporary directory for each test
        use_tmp_dir=True,
        # perform bounded model check
        bmc=True,
        # use 10 iterations for bounded model check
        bmc_depth=10
    )

class Formal_A(FormalBase, entity=ExampleEntity):
    """ a formal testcase that uses the parameters of FormalBase unchanged """

    def architecture(self, dut):
        # some formal properties
        ...

class Formal_B(FormalBase, entity=ExampleEntity):
    """ a formal testcase that overwrites some parameters of FormalBase """

    _yosys_params_ = YosysParams(
        # Use 15 iterations for bounded model check
        # and perform coverage check in addition to bmc.
        # All other settings are inherited from FormalBase.
        cover=True,
        bmc_depth=15
    )

    def architecture(self, dut):
        # some formal properties
        ...